# 데이터 사이언스 특론 7주차 복습 - 통계적 추론 (2) & 경사 하강법

## 목차

1. **p-value와 연속성 보정** (6주차 이어서)
2. **신뢰구간 (Confidence Interval)**
3. **p 해킹 (p-hacking)**
4. **베이즈 추론 (Bayesian Inference)**
5. **경사 하강법 (Gradient Descent)**
6. **경사 하강법으로 모델 학습**
7. **미니배치 경사 하강법 & SGD**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from math import sqrt
import random
from typing import List

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

## 1. p-value와 연속성 보정

### 1.1 p-value 개념

*   **p-value**: 귀무가설이 참이라고 가정했을 때, 실제 관측된 값보다 **더 극단적인 값**이 나올 확률
*   기존의 가설검정에서는 유의수준($\alpha$)에 따라 기각 구간을 미리 설정했지만, p-value 방식은 **관측된 데이터 자체의 극단성**을 직접 측정

**동전 던지기 예시 (양측 검정):**
*   $H_0$: $p = 0.5$ (공평한 동전)
*   1,000번 던져서 앞면이 **530번** 관측
*   정규근사: $\mu = 500$, $\sigma = \sqrt{1000 \times 0.5 \times 0.5} \approx 15.81$
*   연속성 보정 적용: $X_{\text{corrected}} = 529.5$
*   **양측 검정 p-value ≈ 0.062**

**해석:**
*   유의수준 $\alpha = 0.05$로 설정했다면, $0.062 > 0.05$ → **귀무가설을 기각하지 못함**
*   앞면이 530번 나온 것은 공평한 동전에서도 **우연히 일어날 수 있는 수준**이므로, 이 동전이 편향되었다고 단정할 근거가 부족

In [ ]:
# p-value 계산 예시: 동전 1000번 던져서 앞면 530번
n = 1000
p_null = 0.5
mu = n * p_null          # 500
sigma = np.sqrt(n * p_null * (1 - p_null))  # 약 15.81

observed_x = 530
corrected_x = observed_x - 0.5  # 연속성 보정: 530 이상 -> 529.5부터

# sf(survival function) = P(X >= x) = 1 - CDF(x)
p_val_one_sided = stats.norm.sf(corrected_x, loc=mu, scale=sigma)
p_val_two_sided = p_val_one_sided * 2  # 양측 검정

print(f"관측값: {observed_x} (보정 적용: {corrected_x})")
print(f"양측 검정 p-value: {p_val_two_sided:.4f}")
print(f"\n유의수준 5% 기준: {'귀무가설 기각' if p_val_two_sided < 0.05 else '귀무가설 기각 실패'}")
print(f"→ p-value({p_val_two_sided:.3f}) > α(0.05) 이므로 동전이 편향되었다는 근거 부족")

### 1.2 연속성 보정 (Continuity Correction)

*   동전 던지기의 앞면 횟수는 **이항분포** (이산형)를 따르지만, 가설검정에서 **정규분포** (연속형)로 근사
*   이산 → 연속 근사 시 발생하는 오차를 줄이기 위해 **연속성 보정** 적용

**핵심 아이디어:**
*   목표: "앞면이 530번 **이상** 나올 확률"을 구한다
*   이항분포에서 $P(X \geq 530)$은 $X = 530, 531, 532, \ldots$의 확률 합
*   연속형 정규분포에서 이를 적분으로 계산할 때, 적분 구간을 $530$부터 시작하면 $X=530$에 해당하는 확률 막대의 **왼쪽 절반**을 놓침
*   따라서 적분 시작점을 **529.5**로 잡아 보정 → $P(X \geq 529.5)$

**일반 규칙:**
*   $P(X \geq k)$ → $P(X \geq k - 0.5)$
*   $P(X \leq k)$ → $P(X \leq k + 0.5)$
*   $P(X = k)$ → $P(k - 0.5 \leq X \leq k + 0.5)$

In [ ]:
# 연속성 보정 시각화
fig, ax = plt.subplots(figsize=(12, 6))

# 정규분포 곡선
x_range = np.linspace(505, 555, 500)
y_norm = stats.norm.pdf(x_range, loc=mu, scale=sigma)
ax.plot(x_range, y_norm, 'k-', lw=2, label='Normal Approximation')

# 이항분포 PMF (막대)
x_binom = np.arange(505, 556)
y_binom = stats.binom.pmf(x_binom, n, p_null)
ax.bar(x_binom, y_binom, width=1.0, alpha=0.3, color='orange',
       edgecolor='orange', label='Binomial PMF (Discrete)')

# 530 이상 영역 (보정 적용: 529.5부터)
x_fill = np.linspace(529.5, 555, 300)
ax.fill_between(x_fill, stats.norm.pdf(x_fill, mu, sigma),
                alpha=0.4, color='red', label='Normal Integration Area (starts at 529.5)')

# 보정 포인트 표시
ax.axvline(529.5, color='red', ls='-', lw=1.5, label='Correction Point 529.5')
ax.axvline(530, color='blue', ls='--', lw=1.5, label='Discrete Value 530')

ax.set_xlabel('Number of Successes (X)')
ax.set_ylabel('Probability')
ax.set_title('Continuity Correction: Binomial → Normal Approximation')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# 보정 유무에 따른 p-value 비교
p_without = stats.norm.sf(530, loc=mu, scale=sigma) * 2
p_with = stats.norm.sf(529.5, loc=mu, scale=sigma) * 2
p_exact = (1 - stats.binom.cdf(529, n, p_null)) * 2  # 이항분포 정확 계산

print(f"보정 없이 (530부터):   p-value = {p_without:.6f}")
print(f"보정 적용 (529.5부터): p-value = {p_with:.6f}")
print(f"이항분포 정확 계산:     p-value = {p_exact:.6f}")
print(f"\n→ 연속성 보정 적용 시 이항분포의 정확한 값에 더 가까움")

## 2. 신뢰구간 (Confidence Interval)

### 2.1 신뢰구간의 필요성

*   통계적 가설검정에서 **제2종 오류**나 **검정력**을 확인하려면 대립가설 하의 분포를 어느 정도 알고 있어야 함
*   사건에 대한 분포를 **모르는 경우** → **신뢰구간**을 통해 가설 검정

**신뢰구간이란?**
*   모수(parameter)의 참값이 포함될 것으로 기대되는 구간
*   "실험을 수없이 반복하면, 전체 실험의 95%에서는 진짜 파라미터 $p$가 관측된 신뢰구간 안에 존재한다"
*   주의: "$p$가 이 구간에 있을 확률이 95%"가 아님! → $p$는 고정된 값이고, **구간이 확률적**

### 2.2 동전 던지기 예시

*   동전을 1,000번 던져서 앞면이 **525번** 나옴 → $\hat{p} = 0.525$로 추정
*   **중심극한정리** 이용:
    *   $\mu = \hat{p} = 0.525$
    *   $\sigma = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}} = \sqrt{\frac{0.525 \times 0.475}{1000}} \approx 0.0158$
*   **95% 신뢰구간**: $\hat{p} \pm z_{0.025} \cdot \sigma = 0.525 \pm 1.96 \times 0.0158$
    *   → **[0.4940, 0.5560]**
*   $p = 0.5$가 신뢰구간 **안에** 있으므로 → 동전이 공평하지 않다는 결론을 내릴 수 **없음**

**신뢰구간과 가설검정의 관계:**
*   귀무가설의 파라미터 값($p_0 = 0.5$)이 신뢰구간 안에 있으면 → 귀무가설 기각 실패
*   귀무가설의 파라미터 값이 신뢰구간 밖에 있으면 → 귀무가설 기각

In [ ]:
# 신뢰구간 계산: 동전 1000번 던져서 525번 앞면
n = 1000
observed_heads = 525
p_hat = observed_heads / n  # 0.525

# 표준오차 (Standard Error)
se = np.sqrt(p_hat * (1 - p_hat) / n)

# 95% 신뢰구간
z_alpha_half = stats.norm.ppf(0.975)  # ≈ 1.96
ci_lower = p_hat - z_alpha_half * se
ci_upper = p_hat + z_alpha_half * se

print(f"관측 비율 p̂ = {p_hat}")
print(f"표준오차 SE = {se:.4f}")
print(f"z_0.025 = {z_alpha_half:.4f}")
print(f"\n95% 신뢰구간: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"\np=0.5가 신뢰구간 안에 {'있음' if ci_lower <= 0.5 <= ci_upper else '없음'}")
print(f"→ 동전이 공평하지 않다는 결론을 내릴 수 {'없음' if ci_lower <= 0.5 <= ci_upper else '있음'}")

In [ ]:
# 신뢰구간 시각화
fig, ax = plt.subplots(figsize=(10, 5))

x = np.linspace(p_hat - 4*se, p_hat + 4*se, 300)
y = stats.norm.pdf(x, loc=p_hat, scale=se)

ax.plot(x, y, 'b-', lw=2)
ax.fill_between(x, y, where=(x >= ci_lower) & (x <= ci_upper),
                alpha=0.3, color='skyblue', label=f'95% CI [{ci_lower:.4f}, {ci_upper:.4f}]')
ax.axvline(p_hat, color='blue', ls='-', lw=1.5, label=f'p̂ = {p_hat}')
ax.axvline(0.5, color='red', ls='--', lw=2, label='p₀ = 0.5 (H₀)')
ax.axvline(ci_lower, color='green', ls=':', lw=1.5)
ax.axvline(ci_upper, color='green', ls=':', lw=1.5)

ax.set_xlabel('p (probability of heads)')
ax.set_ylabel('Density')
ax.set_title('95% Confidence Interval for Coin Fairness')
ax.legend()
plt.tight_layout()
plt.show()

## 3. p 해킹 (p-hacking)

### 3.1 p 해킹이란?

*   제1종 오류: 유의수준 $\alpha = 0.05$이면 **모든 경우의 5%**에서 귀무가설을 **잘못 기각**하게 됨
*   "의미 있는" 결과를 찾으려고 **여러 차례 다양한 가설**을 검정하다 보면, 그 중 하나는 반드시 유의미해 보이는 결과가 나올 수 있음
*   이렇게 여러 번의 검정을 통해 원하는 p-value를 얻어서 **오판을 맞는 가설이라고 잘못 해석**하는 경우 → **p 해킹**

**직관적 이해:**
*   가설 20개를 동시에 검정하면, 모두 거짓이더라도 **1개 정도**는 $p < 0.05$가 나올 확률이 높음
*   $P(\text{적어도 1개 유의}) = 1 - (1-0.05)^{20} \approx 0.64$

**p 해킹의 구체적 사례:**
*   데이터를 보고 나서 가설을 세우는 행위
*   유의미한 결과가 나올 때까지 변수 조합을 바꿔가며 검정
*   유의미하지 않은 결과는 보고하지 않는 출판 편향 (Publication Bias)

### 3.2 데이터 과학을 위해 추천되는 원칙

1.  **가설은 데이터를 보기 전에 세운다** (Pre-registration)
2.  **데이터를 전처리할 때는 세워둔 가설을 잠시 잊는다** (분석 과정에서 가설에 맞게 데이터를 왜곡하지 않기 위해)
3.  **p-value가 전부는 아니다** → 대안으로 **베이즈 추론** 사용 가능

**추가 대응 방법:**
*   **Bonferroni 보정**: 다중 검정 시 유의수준을 검정 횟수로 나눔 ($\alpha' = \alpha / m$)
*   **효과 크기(Effect Size)** 함께 보고
*   **재현 연구(Replication)** 수행

In [ ]:
# p 해킹 시뮬레이션: 20개의 거짓 가설을 동시에 검정
np.random.seed(42)
n_tests = 20
n_simulations = 10000
alpha = 0.05

false_positive_count = 0

for _ in range(n_simulations):
    # 각 시뮬레이션에서 20개의 "효과 없는" 가설 검정
    p_values = []
    for _ in range(n_tests):
        # 귀무가설이 참인 데이터 (두 그룹 모두 N(0,1)에서 추출)
        group_a = np.random.normal(0, 1, 30)
        group_b = np.random.normal(0, 1, 30)
        _, p_val = stats.ttest_ind(group_a, group_b)
        p_values.append(p_val)
    
    # 20개 중 하나라도 유의미한 결과가 나왔는지?
    if min(p_values) < alpha:
        false_positive_count += 1

prob_false_positive = false_positive_count / n_simulations
theoretical = 1 - (1 - alpha) ** n_tests

print(f"가설 {n_tests}개를 동시 검정 (모두 효과 없음)")
print(f"시뮬레이션 결과: 적어도 1개 유의미한 결과가 나올 확률 = {prob_false_positive:.4f}")
print(f"이론적 확률: 1 - (1-0.05)^20 = {theoretical:.4f}")
print(f"\n→ 효과가 전혀 없어도 {prob_false_positive*100:.1f}%의 경우에서 '유의미한' 결과를 찾게 됨!")

## 4. 베이즈 추론 (Bayesian Inference)

### 4.1 통계적 추론 vs 베이즈 추론

| 구분 | 통계적 추론 (빈도주의) | 베이즈 추론 |
|------|----------------------|------------|
| **관점** | 확률 = 장기적 빈도 | 확률 = 믿음의 정도 |
| **파라미터** | 고정된 미지의 값 | **확률변수**로 취급 |
| **결론** | "귀무가설이 참이라면, 이런 극단적 데이터가 나올 확률은 3%" | "관측 데이터를 고려하면, 파라미터가 이 범위에 있을 확률은 95%" |
| **핵심 도구** | p-value, 신뢰구간 | 사전분포 → 사후분포 갱신 |
| **장점** | 객관적, 사전 정보 불필요 | 직관적 해석, 순차적 갱신 가능 |
| **단점** | 해석이 반직관적 | 사전분포 선택이 주관적, 수학적으로 복잡 |

### 4.2 베이즈 추론의 핵심 아이디어

*   알려지지 않은 파라미터를 **확률변수**로 봄
*   **사전분포 (Prior)**: 데이터를 보기 전, 파라미터에 대한 사전 믿음
*   **가능도 (Likelihood)**: 주어진 파라미터 하에서 관측 데이터가 나올 확률
*   **사후분포 (Posterior)**: 데이터를 관측한 후 갱신된 파라미터에 대한 믿음

$$P(\theta | \text{data}) = \frac{P(\text{data} | \theta) \cdot P(\theta)}{P(\text{data})}$$

$$\text{Posterior} \propto \text{Likelihood} \times \text{Prior}$$

### 4.3 베타분포 (Beta Distribution)

*   확률값 $p \in [0, 1]$에 대한 분포로, **베이즈 추론에서 사전분포로 자주 사용**
*   PDF: $f(x; \alpha, \beta) = \frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha, \beta)}$
*   여기서 $B(\alpha, \beta)$는 베타함수 (정규화 상수)

**베타분포의 성질:**
*   **평균(중심)**: $\frac{\alpha}{\alpha + \beta}$
*   $\alpha, \beta$가 **클수록** 분포가 더 **중심으로 몰림** (= 확신이 강함)
*   $\alpha = \beta = 1$이면 **균등분포** (= 아무 사전 정보 없음)
*   $\alpha > \beta$이면 분포가 **오른쪽으로 치우침** (높은 확률에 대한 믿음)
*   $\alpha < \beta$이면 분포가 **왼쪽으로 치우침** (낮은 확률에 대한 믿음)

In [ ]:
# 베타분포 시각화: 다양한 α, β 조합
fig, ax = plt.subplots(figsize=(10, 6))
x = np.linspace(0.001, 0.999, 300)

params = [
    (1, 1, 'k', '-'),    # 균등분포 (무정보)
    (10, 10, 'b', '-'),  # 0.5 중심, 적당한 확신
    (4, 16, 'r', '-'),   # 0.2 중심, 낮은 확률 믿음
    (16, 3, 'g', '-'),   # 0.84 중심, 높은 확률 믿음
]

for alpha_p, beta_p, color, ls in params:
    y = stats.beta.pdf(x, alpha_p, beta_p)
    mean_val = alpha_p / (alpha_p + beta_p)
    ax.plot(x, y, color=color, ls=ls, lw=2,
            label=f'α={alpha_p}, β={beta_p} (Mean: {mean_val:.2f})')

ax.set_xlabel('x (Probability)')
ax.set_ylabel('Probability Density')
ax.set_title('Beta Distribution with Various Parameters')
ax.legend(fontsize=10)
ax.set_ylim(0, 6)
plt.tight_layout()
plt.show()

### 4.4 베이즈 추론 - 동전 던지기 예시

**이항분포 + 베타분포의 켤레 사전분포 (Conjugate Prior) 관계:**

*   사전분포: $p \sim \text{Beta}(\alpha, \beta)$
*   관측 데이터: 앞면 $h$번, 뒷면 $t$번
*   **사후분포**: $p | \text{data} \sim \text{Beta}(\alpha + h, \beta + t)$

→ 사전분포의 $\alpha$에 앞면 수를 더하고, $\beta$에 뒷면 수를 더하면 사후분포가 됨!

**동전을 10번 던져서 앞면 3번 관측 ($h=3, t=7$):**

| 사전분포 | 사전 믿음 | 사후분포 | 사후 중심 | 해석 |
|----------|----------|----------|----------|------|
| $\text{Beta}(1, 1)$ | 균등 (무정보) | $\text{Beta}(4, 8)$ | 0.33 | 데이터에 크게 의존 |
| $\text{Beta}(20, 20)$ | 공평하다는 강한 믿음 | $\text{Beta}(23, 27)$ | 0.46 | 뒷면 약간 편향으로 믿음 변경 |
| $\text{Beta}(30, 10)$ | 앞면 편향 강한 믿음 | $\text{Beta}(33, 17)$ | 0.66 | 0.75→0.66, 앞면 편향 믿음 약화 |

In [ ]:
# 베이즈 갱신 시각화: 동전 10번 던져서 앞면 3번
h, t = 3, 7  # 앞면 3번, 뒷면 7번

priors = [
    (1, 1, 'Uniform (no info)'),
    (20, 20, 'Fair coin belief'),
    (30, 10, 'Heads-biased belief'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.linspace(0.001, 0.999, 300)
colors_prior = ['lightcoral', 'lightblue', 'lightgreen']
colors_post = ['red', 'blue', 'green']

for ax, (a, b, label), cp, cpo in zip(axes, priors, colors_prior, colors_post):
    # 사전분포
    y_prior = stats.beta.pdf(x, a, b)
    ax.plot(x, y_prior, '--', color=cpo, lw=1.5, alpha=0.6,
            label=f'Prior: Beta({a},{b}), mean={a/(a+b):.2f}')
    ax.fill_between(x, y_prior, alpha=0.1, color=cp)
    
    # 사후분포
    a_post, b_post = a + h, b + t
    y_post = stats.beta.pdf(x, a_post, b_post)
    ax.plot(x, y_post, '-', color=cpo, lw=2.5,
            label=f'Posterior: Beta({a_post},{b_post}), mean={a_post/(a_post+b_post):.2f}')
    ax.fill_between(x, y_post, alpha=0.2, color=cpo)
    
    ax.axvline(0.5, color='gray', ls=':', lw=1, alpha=0.5)
    ax.set_xlabel('p (Probability of Heads)')
    ax.set_ylabel('Density')
    ax.set_title(f'Prior: {label}')
    ax.legend(fontsize=8)

fig.suptitle('Bayesian Update: 10 flips, 3 heads, 7 tails', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 세 사후분포를 한 그래프에 겹쳐서 비교
fig, ax = plt.subplots(figsize=(10, 6))
x = np.linspace(0.001, 0.999, 300)

posteriors = [
    (4, 8, 'r', 'Beta(4,8)'),
    (23, 27, 'b', 'Beta(23,27)'),
    (33, 17, 'g', 'Beta(33,17)'),
]

for a, b, color, label in posteriors:
    mean_val = a / (a + b)
    y = stats.beta.pdf(x, a, b)
    ax.plot(x, y, color=color, lw=2.5,
            label=f'α={a}, β={b} (Mean: {mean_val:.2f})')
    ax.fill_between(x, y, alpha=0.15, color=color)

ax.axvline(0.5, color='gray', ls='--', lw=1, alpha=0.5, label='p=0.5')
ax.set_xlabel('Probability (x)')
ax.set_ylabel('Probability Density')
ax.set_title('Posterior Distributions After Observing 3H, 7T')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 4.5 데이터가 많아지면 사전분포의 영향 감소

*   동전을 **더 많이 던져볼수록** 사전분포는 의미가 없어지고, **사전분포에 상관없이 거의 동일한 사후분포**가 계산됨
*   e.g., 아무리 사전에 동전이 편향되었다고 믿었더라도, 동전을 **2,000번** 던져서 앞면이 **1,000번** 나왔다면 → 생각을 바꿀 수밖에 없음

**베타분포의 정규근사:**
*   충분히 큰 $\alpha, \beta$에 대해:
$$\text{Beta}(\alpha, \beta) \approx N\left(\frac{\alpha}{\alpha+\beta}, \sqrt{\frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}}\right)$$

**베이즈 추론의 결론 방식:**
*   "관측된 데이터와 사전분포를 고려했을 때, 동전의 앞면이 나올 확률이 49%~51%일 경우는 5%밖에 되지 않는다."
*   vs. 통계적 추론: "동전이 공평하다면 이렇게 편향된 데이터를 관측할 경우는 5%밖에 되지 않는다."

**베이즈 추론의 단점:**
*   수학적으로 복잡해짐 (해석적 해가 존재하지 않는 경우 MCMC 등 수치적 방법 필요)
*   **사전분포의 선택이 주관적** → 분석자에 따라 결과가 달라질 수 있음

In [ ]:
# 데이터가 많아질수록 사전분포의 영향이 줄어드는 것을 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.linspace(0.35, 0.65, 300)

# 서로 다른 사전분포 3개
priors = [(1, 1, 'Uniform'), (30, 10, 'Heads-biased'), (5, 30, 'Tails-biased')]
colors = ['red', 'green', 'blue']

# 관측: 실제 p=0.5인 동전에서 n번 던져서 절반 앞면
data_sizes = [(10, 5), (100, 50), (2000, 1000)]

for ax, (n_flips, n_heads) in zip(axes, data_sizes):
    n_tails = n_flips - n_heads
    for (a, b, label), color in zip(priors, colors):
        a_post = a + n_heads
        b_post = b + n_tails
        y = stats.beta.pdf(x, a_post, b_post)
        mean_val = a_post / (a_post + b_post)
        ax.plot(x, y, color=color, lw=2,
                label=f'{label} → mean={mean_val:.3f}')
    
    ax.axvline(0.5, color='gray', ls='--', alpha=0.5)
    ax.set_title(f'After {n_flips} flips ({n_heads}H, {n_tails}T)')
    ax.set_xlabel('p')
    ax.legend(fontsize=8)

fig.suptitle('Prior Influence Diminishes with More Data', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("→ n=2000이 되면 어떤 사전분포로 시작해도 사후분포가 거의 동일해짐!")

## 5. 경사 하강법 (Gradient Descent)

### 5.1 경사 하강법의 필요성

*   데이터 과학에서는 특정 상황에 **가장 적합한 모델**을 찾아야 할 때가 많음
*   "가장 적합한(best)"이란 대부분:
    *   모델의 **오류(error)를 최소화**하는 것, 또는
    *   **가능성(likelihood)을 최대화**하는 것
    *   → 주어진 **최적화 문제**에 관한 답을 내리는 것

**경사 하강법이 필요한 이유:**
*   확률 분포를 정확하게 모르거나, 수학적으로 수식이 너무 복잡하여 **closed-form solution**을 구하지 못하는 경우
*   이때 **수치적인 방법**으로 오차를 줄여 나가며 최적화 → **경사 하강법**
*   ML/DL 등 AI 분야뿐만 아니라, **convex optimization** 등 최적화 이론에서도 가장 많이 사용

### 5.2 경사 하강법의 핵심 원리

*   실수 벡터를 입력하면 실수 하나를 출력해주는 함수 $f$가 있다고 가정
*   **목적**: 이 함수를 최소화(또는 최대화)시키는 **입력값**을 찾기
*   **그래디언트(gradient)**: 함수가 **가장 빠르게 증가하는 방향**을 가리키는 벡터

**최솟값을 찾는 알고리즘:**
1.  임의의 시작점 $\theta_0$을 잡음
2.  현재 위치에서 gradient를 계산: $\nabla f(\theta)$
3.  gradient의 **반대 방향**으로 조금씩 이동: $\theta_{t+1} = \theta_t - \eta \nabla f(\theta_t)$
    *   여기서 $\eta$는 **학습률 (learning rate)** = 이동 거리
4.  수렴할 때까지 2~3 반복

**gradient 계산 방법:**
*   **해석적**: 미분값, 편도함수값으로 직접 계산
*   **수치적**: 아주 작은 값 $h$를 이용하여 도함수의 극한 정의로 근사
    *   $\frac{\partial f}{\partial \theta_i} \approx \frac{f(\theta + h \cdot e_i) - f(\theta)}{h}$

**주의: Local Minimum 문제**
*   함수에 여러 개의 local minimum이 있다면 → 시작점에 따라 **전역 최솟값(global minimum)을 찾지 못할 수 있음**

In [ ]:
# 경사 하강법 1D 시각화
def f(x):
    return (x - 3)**2 + 2

def f_grad(x):
    return 2 * (x - 3)

# 경사 하강법 실행
x_current = -2.0
lr = 0.1
trajectory = [x_current]

for _ in range(30):
    grad = f_grad(x_current)
    x_current = x_current - lr * grad
    trajectory.append(x_current)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))
x = np.linspace(-4, 8, 300)
ax.plot(x, f(x), 'b-', lw=2, label='f(x) = (x-3)² + 2')

traj = np.array(trajectory)
ax.plot(traj, f(traj), 'ro-', markersize=6, lw=1.5, alpha=0.7,
        label='Gradient Descent Path')
ax.plot(traj[0], f(traj[0]), 'gs', markersize=12, label=f'Start (x={traj[0]:.1f})')
ax.plot(traj[-1], f(traj[-1]), 'r*', markersize=15, label=f'End (x={traj[-1]:.3f})')

ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('1D Gradient Descent Example')
ax.legend()
plt.tight_layout()
plt.show()

### 5.3 적절한 이동 거리 (Learning Rate) 정하기

**이동 거리를 정하는 세 가지 방법:**
1.  **고정** (Fixed step size): 단순하지만 최적이 아닐 수 있음
2.  **시간에 따라 점차 줄임** (Learning rate decay): 처음에는 크게, 나중에는 섬세하게
3.  **매 스텝마다 목적함수를 최소화하는 이동 거리로 설정** (Line search): 가장 이상적이지만 **계산 비용이 너무 큼**

**실무에서의 접근:**
*   과학적/공학적으로 정확히 계산되는 값이 아님
*   여러 값으로 **실험을 해보고** 시행착오를 통해 **경험적인 값**으로 사용
*   문제에 따라 그 연구자의 **노하우**라고도 함

In [ ]:
# Learning Rate에 따른 경사 하강법 동작 비교
def f(x): return (x - 3)**2 + 2
def f_grad(x): return 2 * (x - 3)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
learning_rates = [0.01, 0.3, 0.95]
titles = ['Too Small (lr=0.01)', 'Good (lr=0.3)', 'Too Large (lr=0.95)']

for ax, lr, title in zip(axes, learning_rates, titles):
    x_range = np.linspace(-4, 10, 300)
    ax.plot(x_range, f(x_range), 'b-', lw=2)
    
    x_curr = -2.0
    traj = [x_curr]
    for _ in range(20):
        x_curr = x_curr - lr * f_grad(x_curr)
        traj.append(x_curr)
    
    traj = np.array(traj)
    ax.plot(traj, f(traj), 'ro-', markersize=5, lw=1, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.set_ylim(-5, 50)

plt.suptitle('Effect of Learning Rate on Gradient Descent', fontsize=14)
plt.tight_layout()
plt.show()

print("lr이 너무 작으면: 수렴이 매우 느림")
print("lr이 적절하면: 빠르고 안정적으로 수렴")
print("lr이 너무 크면: 진동하거나 발산할 수 있음")

## 6. 경사 하강법으로 모델 학습

### 6.1 손실 함수와 모델 파라미터

*   주어진 데이터가 변하지 않는다고 가정하면, **손실 함수(loss function)**는 모델의 **파라미터가 얼마나 좋고 나쁜지** 알려줌
*   경사 하강법으로 **손실을 최소화하는 모델의 파라미터**를 구할 수 있음

**간단한 예제: 선형 회귀**
*   $x$와 $y$가 선형 관계라는 것을 이미 알고 있음
*   Ground Truth: $y = 20x + 5$
*   모델: $\hat{y} = \text{slope} \cdot x + \text{intercept}$
*   **목표**: 경사 하강법으로 **평균제곱오차(MSE)**를 최소화하여 $\text{slope} \approx 20$, $\text{intercept} \approx 5$를 찾기

### 6.2 Gradient 유도

**한 개의 데이터 포인트 $(x, y)$에서:**

*   오차: $e = (\text{slope} \cdot x + \text{intercept}) - y$
*   손실: $L = e^2 = (\text{slope} \cdot x + \text{intercept} - y)^2$

**편미분 (Chain Rule 적용):**

$$\frac{\partial L}{\partial \text{slope}} = 2e \cdot \frac{\partial e}{\partial \text{slope}} = 2e \cdot x$$

$$\frac{\partial L}{\partial \text{intercept}} = 2e \cdot \frac{\partial e}{\partial \text{intercept}} = 2e \cdot 1$$

**평균제곱오차(MSE)로 확장:**

$$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

*   MSE의 gradient = 각 데이터 포인트에서 계산된 gradient의 **평균**

In [ ]:
# 데이터 생성: y = 20x + 5
# x는 -50~49 사이의 값
inputs = [(x, x * 20 + 5) for x in range(-50, 50)]

print(f"데이터 수: {len(inputs)}")
print(f"처음 5개: {inputs[:5]}")
print(f"마지막 5개: {inputs[-5:]}")

In [ ]:
# 한 개의 데이터 포인트에서 오차의 gradient를 계산하는 함수
Vector = List[float]

def linear_gradient(x: float, y: float, theta: Vector) -> Vector:
    slope, intercept = theta
    predicted = slope * x + intercept  # 모델의 예측값
    error = predicted - y              # 오차 = 예측값 - 실제값
    # squared_error = error ** 2       # 오차의 제곱을 최소화
    grad = [2 * error * x,             # ∂L/∂slope = 2e * x
            2 * error]                 # ∂L/∂intercept = 2e * 1
    return grad

# 테스트: theta = [0, 0]일 때 (x=1, y=25)에서의 gradient
test_grad = linear_gradient(1, 25, [0, 0])
print(f"theta=[0,0], (x=1, y=25): gradient = {test_grad}")
print(f"  → error = 0*1+0 - 25 = -25")
print(f"  → ∂L/∂slope = 2*(-25)*1 = {2*(-25)*1}")
print(f"  → ∂L/∂intercept = 2*(-25) = {2*(-25)}")

### 6.3 배치 경사 하강법 (Batch Gradient Descent)

**절차:**
1.  임의의 theta로 시작
2.  **모든** 데이터 포인트에서 gradient 계산 후 **평균**
3.  theta를 gradient 방향의 반대로 업데이트
4.  반복

*   **에폭(epoch)**: 전체 데이터셋을 한 번 훑어보는 것
*   여러 에폭을 수행하면 올바른 slope와 intercept가 학습됨

In [ ]:
# 배치 경사 하강법 (Batch Gradient Descent)
import random

# 초기 파라미터 및 하이퍼파라미터 설정
random.seed(42)
theta = [random.uniform(-1, 1), random.uniform(-1, 1)]  # [slope, intercept]
learning_rate = 0.001

history = {'epoch': [], 'slope': [], 'intercept': [], 'mse': []}

for epoch in range(5000):
    # (1) 모든 데이터 포인트에 대한 그래디언트 리스트 생성
    gradients = [linear_gradient(x, y, theta) for x, y in inputs]
    
    # (2) gradient 평균 (vector_mean)
    n = len(gradients)
    mean_dw = sum(g[0] for g in gradients) / n
    mean_db = sum(g[1] for g in gradients) / n
    mean_grad = [mean_dw, mean_db]
    
    # (3) 파라미터 업데이트 (theta = theta - learning_rate * mean_grad)
    theta = [t - learning_rate * g for t, g in zip(theta, mean_grad)]
    
    # MSE 계산
    mse = sum((theta[0]*x + theta[1] - y)**2 for x, y in inputs) / len(inputs)
    
    if epoch % 500 == 0:
        print(f"Epoch {epoch:4d} : theta = [{theta[0]:.6f}, {theta[1]:.6f}], MSE = {mse:.6f}")
        history['epoch'].append(epoch)
        history['slope'].append(theta[0])
        history['intercept'].append(theta[1])
        history['mse'].append(mse)

print(f"\n최종 결과: slope = {theta[0]:.10f}, intercept = {theta[1]:.10f}")
print(f"Ground Truth: slope = 20, intercept = 5")

In [ ]:
# 학습 과정 시각화: MSE 감소 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE 변화
axes[0].plot(history['epoch'], history['mse'], 'b-o', lw=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].set_title('Training Loss (MSE) over Epochs')
axes[0].set_yscale('log')

# 파라미터 수렴
axes[1].plot(history['epoch'], history['slope'], 'r-o', lw=2, label='slope (→ 20)')
axes[1].plot(history['epoch'], history['intercept'], 'g-s', lw=2, label='intercept (→ 5)')
axes[1].axhline(20, color='r', ls='--', alpha=0.5)
axes[1].axhline(5, color='g', ls='--', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Parameter Value')
axes[1].set_title('Parameter Convergence')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. 미니배치 경사 하강법 & SGD

### 7.1 배치 경사 하강법의 단점

*   매 에폭마다 **데이터셋 전체**의 gradient를 모두 구해야 함
*   데이터셋이 커지면 gradient를 모두 구하는 것이 **매우 오래 걸림**
*   → **더 자주** gradient만큼 이동하는 방법이 필요

### 7.2 미니배치 경사 하강법 (Minibatch Gradient Descent)

*   전체 데이터셋의 **샘플(미니배치)**에서 gradient를 계산하여 업데이트
*   배치 크기 (batch_size)는 하이퍼파라미터 (보통 16, 32, 64, 128 등)

**미니배치 절차:**
1.  에폭 시작 시 데이터를 **무작위로 섞음** (shuffle)
2.  batch_size만큼 데이터 슬라이싱
3.  해당 미니배치에서 gradient 평균 계산
4.  파라미터 업데이트 (**매 미니배치마다**)
5.  다음 미니배치로 이동, 에폭 끝나면 1로 돌아감

In [ ]:
# 미니배치 경사 하강법
random.seed(42)
theta_mini = [random.uniform(-1, 1), random.uniform(-1, 1)]
learning_rate = 0.001
batch_size = 20  # 미니배치 사이즈

inputs_copy = inputs.copy()

for epoch in range(1000):
    # (1) 에폭 시작 시 데이터를 무작위로 섞음 (무작위성 부여)
    random.shuffle(inputs_copy)
    
    # (2) 미니배치 단위로 루프 수행
    for i in range(0, len(inputs_copy), batch_size):
        # batch_size만큼 데이터 슬라이싱
        batch = inputs_copy[i : i + batch_size]
        
        # 해당 미니배치에 대한 그래디언트 계산
        batch_gradients = [linear_gradient(x, y, theta_mini) for x, y in batch]
        
        # 미니배치 그래디언트의 평균 계산
        n_batch = len(batch_gradients)
        mean_dw = sum(g[0] for g in batch_gradients) / n_batch
        mean_db = sum(g[1] for g in batch_gradients) / n_batch
        mean_grad = [mean_dw, mean_db]
        
        # 파라미터 업데이트 (매 미니배치마다 업데이트 발생)
        theta_mini = [t - learning_rate * g for t, g in zip(theta_mini, mean_grad)]
    
    if epoch % 100 == 0:
        mse = sum((theta_mini[0]*x + theta_mini[1] - y)**2 for x, y in inputs) / len(inputs)
        print(f"Epoch {epoch:4d} : theta = [{theta_mini[0]:.6f}, {theta_mini[1]:.6f}], MSE = {mse:.6f}")

print(f"\n최종 결과: slope = {theta_mini[0]:.10f}, intercept = {theta_mini[1]:.10f}")

### 7.3 SGD (Stochastic Gradient Descent)

*   **극단적인 미니배치**: 각 에폭마다 **단 하나의 데이터 포인트**에서 gradient를 계산하여 즉시 업데이트
*   에폭 당 업데이트가 데이터 개수만큼 발생하므로, **훨씬 더 적은 에폭** 안에서 최적의 파라미터를 찾을 수 있음

**SGD의 장단점:**
*   **장점**: 빠른 수렴, 계산 효율적
*   **단점**:
    *   한 데이터마다 gradient를 계산하므로 **학습이 불안정**
    *   learning rate에 따라 **학습이 수렴하지 않을 수 있음**
    *   특정 데이터 포인트의 gradient와 전체의 gradient 방향이 **상반**될 수 있음

### 7.4 SGD - Gradient Explosion 문제

*   일부 데이터 표본에서 (e.g., $x=50$일 때) gradient가 **너무 커서**, learning rate를 곱해줘도 한 번의 업데이트로 파라미터를 **너무 멀리 밀어버림**
*   파라미터 값이 점점 커지면서 발산 → **Gradient Explosion**

**해결법:**
*   **Learning rate를 더 작게** 설정 (e.g., 0.001 → 0.0001)
*   **Gradient Clipping**: gradient의 크기를 일정 값으로 제한
*   **적응형 학습률** (Adam, RMSProp 등): 파라미터별로 학습률을 자동 조절

In [ ]:
# SGD: lr=0.001에서 Gradient Explosion 발생 데모
random.seed(42)
theta_sgd_bad = [random.uniform(-1, 1), random.uniform(-1, 1)]
learning_rate_bad = 0.001
inputs_copy = inputs.copy()

print("=== SGD with lr=0.001 (Gradient Explosion 발생) ===")
for epoch in range(100):
    random.shuffle(inputs_copy)
    for x, y in inputs_copy:
        grad = linear_gradient(x, y, theta_sgd_bad)
        theta_sgd_bad = [t - learning_rate_bad * g for t, g in zip(theta_sgd_bad, grad)]
    
    if epoch % 10 == 0:
        # overflow 방지를 위한 체크
        if abs(theta_sgd_bad[0]) > 1e15:
            print(f"Epoch {epoch:3d} : theta = [{theta_sgd_bad[0]:.2e}, {theta_sgd_bad[1]:.2e}] ← EXPLODED!")
        else:
            print(f"Epoch {epoch:3d} : theta = [{theta_sgd_bad[0]:.4f}, {theta_sgd_bad[1]:.4f}]")

In [ ]:
# SGD: lr=0.0001로 줄여서 안정적 학습
random.seed(42)
theta_sgd = [random.uniform(-1, 1), random.uniform(-1, 1)]
learning_rate_sgd = 0.0001  # Gradient Explosion 방지를 위해 lr 축소
inputs_copy = inputs.copy()

sgd_history = []

print("=== SGD with lr=0.0001 (안정적 학습) ===")
for epoch in range(200):
    random.shuffle(inputs_copy)
    for x, y in inputs_copy:
        grad = linear_gradient(x, y, theta_sgd)
        theta_sgd = [t - learning_rate_sgd * g for t, g in zip(theta_sgd, grad)]
    
    mse = sum((theta_sgd[0]*x + theta_sgd[1] - y)**2 for x, y in inputs) / len(inputs)
    sgd_history.append(mse)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} : theta = [{theta_sgd[0]:.6f}, {theta_sgd[1]:.6f}], MSE = {mse:.4f}")

print(f"\n최종 결과: slope = {theta_sgd[0]:.10f}, intercept = {theta_sgd[1]:.10f}")

In [ ]:
# SGD MSE 변화 시각화
plt.figure(figsize=(10, 5))
plt.plot(sgd_history, 'b-', lw=1.5, alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('SGD Training Loss (lr=0.0001)')
plt.yscale('log')
plt.tight_layout()
plt.show()

### 7.5 세 가지 경사 하강법 비교 정리

| 구분 | Batch GD | Mini-batch GD | SGD |
|------|----------|---------------|-----|
| **gradient 계산 단위** | 전체 데이터셋 | 미니배치 (e.g., 20개) | 데이터 1개 |
| **에폭 당 업데이트 횟수** | 1회 | 데이터수/배치크기 | 데이터 수 |
| **수렴 안정성** | 가장 안정적 | 중간 | 가장 불안정 |
| **계산 비용 (에폭 당)** | 높음 | 중간 | 낮음 |
| **수렴 속도 (에폭 수)** | 느림 | 중간 | 빠름 (단, 진동) |
| **Gradient Explosion** | 위험 낮음 | 위험 낮음 | **위험 높음** |

**실무에서는 보통 미니배치 경사 하강법을 사용**하며, 배치를 **벡터화(vectorization)**하여 GPU로 효율적으로 계산

### 7.6 실제 ML/최적화에서의 추가 고려사항

*   **Ground truth를 모르는 경우가 많음**: 오차를 비교할 실제 정답 자체를 알지 못하는 경우가 대부분
*   **Non-convex 문제**: 최소화하는 경우, 데이터셋 전체에 대해 **convex가 아닐 수도** 있음 → local minimum에 빠질 위험
*   **일반화 (Generalization)**: 보지 못한 데이터, 전체 모집단에 대해 일반화가 필요 → 현재 데이터만으로 **overfitting**에 빠질 수 있음
*   **데이터의 질과 노이즈**: 노이즈가 많은 데이터는 gradient 추정을 부정확하게 만듦
*   **파라미터 수**: 실제 딥러닝 모델은 **수백만~수십억 개**의 파라미터를 가짐

## 정리: 7주차 핵심 개념 요약

| 주제 | 핵심 내용 |
|------|----------|
| **p-value** | 귀무가설이 참일 때 관측값보다 극단적인 결과가 나올 확률. α보다 작으면 기각 |
| **연속성 보정** | 이산분포 → 연속분포 근사 시 ±0.5 보정으로 정확도 향상 |
| **신뢰구간** | 대립분포를 모를 때 사용. 귀무가설 값이 구간 안에 있으면 기각 실패 |
| **p 해킹** | 다중 검정으로 거짓 유의미 결과를 얻는 문제. 사전 가설 설정이 핵심 |
| **베이즈 추론** | 사전분포 + 데이터 → 사후분포 갱신. 파라미터 자체에 확률적 결론 가능 |
| **베타분포** | [0,1]에서 정의, 이항분포의 켤레 사전분포. α/(α+β)가 중심 |
| **경사 하강법** | gradient의 반대 방향으로 이동하여 손실 최소화. lr 선택이 중요 |
| **Batch/Mini-batch/SGD** | 전체/미니배치/1개 데이터로 gradient 계산. 안정성과 속도의 trade-off |